In [21]:
from POSEIDON.constants import R_Sun, R_J, R_E, M_E
from POSEIDON.core import create_star, create_planet, load_data, define_model, \
                          wl_grid_constant_R, set_priors, read_opacities, make_atmosphere, \
                          compute_spectrum
from POSEIDON.visuals import plot_data, plot_spectra_retrieved, plot_PT_retrieved, \
                             plot_chem_retrieved, plot_spectra
from POSEIDON.retrieval import run_retrieval
from POSEIDON.utility import read_retrieved_spectrum, read_retrieved_PT, \
                             read_retrieved_log_X, plot_collection, write_spectrum
from POSEIDON.corner import generate_cornerplot
from POSEIDON.instrument import generate_syn_data_from_file

import numpy as np
import matplotlib as plt

from scipy.constants import parsec as pc

In [2]:
#***** Model wavelength grid *****#

wl_min = 0.58      # Minimum wavelength (um)           2.8
wl_max = 5.30      # Maximum wavelength (um)           5.3
R = 20000          # Spectral resolution of grid

# We need to provide a model wavelength grid to initialise instrument properties
wl = wl_grid_constant_R(wl_min, wl_max, R)

#***** Define stellar properties *****#

R_s = 0.38*R_Sun      # Stellar radius (m)
T_s = 3506.0          # Stellar effective temperature (K)
err_T_s = 70          # Value in ExoMast
Met_s = -0.20         # Stellar metallicity [log10(Fe/H_star / Fe/H_solar)]
log_g_s = 4.872       # Stellar log surface gravity (log10(cm/s^2) by convention)


# Create the stellar object
# star = create_star(R_s, T_s, log_g_s, Met_s)
star = create_star(R_s, T_s, log_g_s, Met_s, T_eff_error = err_T_s, wl = wl)

In [18]:
#***** Define planet properties *****#

# planet_name = 'TOI-270e'  # Planet name used for plots, output files etc.
planet_names = ['TOI-270e','TOI-270f','TOI-270g','TOI-270h','TOI-270i','TOI-270j','TOI-270k','TOI-270l','TOI-270m','TOI-270n',]  # Planet names used for plots, output files etc.
planets = []
colour_list = ['green', 'red', 'black', 'darkgrey', 'navy', 'brown', 'goldenrod', 'magenta', 'cyan', 'orange']
list_of_spectra = []

R_p = 2.133*R_E     # Planetary radius (m)
# M_p = 4.78*M_E      # Planet mass
masses = np.logspace(np.log10(2),np.log10(20),20)
M_p = masses*M_E
T_eq = 387.8       # Equilibrium temperature (K)
d = 22.453*pc       # Distance to system (m)

masses

array([ 2.        ,  2.25767578,  2.54854997,  2.87689978,  3.24755348,
        3.66596142,  4.13827616,  4.67144294,  5.2733018 ,  5.95270288,
        6.71963657,  7.58538038,  8.5626648 ,  9.66586048, 10.91118956,
       12.31696422, 13.90385592, 15.69519941, 17.71733581, 20.        ])

In [40]:
for i in range(0,len(M_p)):
    # print(i)
    # Create the planet object
    # planet = create_planet(planet_name, R_p, mass = M_p, gravity = g_p, T_eq = T_eq)
    planet = create_planet(planet_names[i], R_p, mass = M_p[i], T_eq = T_eq, d = d)
    planets.append(planet)

    #***** Define model *****#

    model_name = 'H2_H2O_Forward_Model_' + str(i+1)  # Model name used for plots, output files etc.

    bulk_species = ['H2', 'He']      # H2 + He comprises the bulk atmosphere
    param_species = ['H2O']   # The trace gases are H2O and CH4

    # Create the model object
    model = define_model(model_name, bulk_species, param_species, 
                        PT_profile = 'isotherm', cloud_model = 'cloud-free',
                        radius_unit = 'R_E', surface = False,
                        stellar_contam = None)


                        # Specify the pressure grid of the atmosphere
    P_min = 1.0e-7    # 0.1 ubar
    P_max = 100       # 100 bar
    N_layers = 100    # 100 layers

    # We'll space the layers uniformly in log-pressure
    P = np.logspace(np.log10(P_max), np.log10(P_min), N_layers)

    # Specify the reference pressure and radius
    P_ref = 10.0   # Reference pressure (bar)
    R_p_ref = R_p  # Radius at reference pressure

    # Provide a specific set of model parameters for the atmosphere 
    PT_params = np.array([1000])              # T (K)
    # log_X_params = np.array([-3.3])     # log(H2O)
    # log_X_params = np.array([-0.0004148557908770917, -0.0004148557908770917, -0.12483854208890861, -0.5075795689696349, -1.001908614024893,
    #                          -1.640359126573138, -2.464949672183401, -3.529949315140789, -4.905449247243431, -6.681975723746822])     # log(H2O)
    log_X_params = np.array([[-0.0004148557908770917], [-0.0004148557908770917], [-0.12483854208890861], [-0.5075795689696349], [-1.001908614024893],
                             [-1.640359126573138], [-2.464949672183401], [-3.529949315140789], [-4.905449247243431], [-6.681975723746822]])     # log(H2O)


    # Generate the atmosphere
    atmosphere = make_atmosphere(planet, model, P, P_ref, R_p_ref, 
                                PT_params, log_X_params[i])



In [ ]:
for i in range(0,len(M_p)):
 
    if i == 0:
        #***** Read opacity data *****#

        opacity_treatment = 'opacity_sampling'

        # First, specify limits of the fine temperature and pressure grids for the 
        # pre-interpolation of cross sections. These fine grids should cover a
        # wide range of possible temperatures and pressures for the model atmosphere.

        # Define fine temperature grid (K)
        T_fine_min = 100     # 400 K lower limit suffices for a typical hot Jupiter
        T_fine_max = 1000    # 2000 K upper limit suffices for a typical hot Jupiter
        T_fine_step = 10     # 10 K steps are a good tradeoff between accuracy and RAM

        T_fine = np.arange(T_fine_min, (T_fine_max + T_fine_step), T_fine_step)

        # Define fine pressure grid (log10(P/bar))
        log_P_fine_min = -6.0   # 1 ubar is the lowest pressure in the opacity database
        log_P_fine_max = 2.0    # 100 bar is the highest pressure in the opacity database
        log_P_fine_step = 0.2   # 0.2 dex steps are a good tradeoff between accuracy and RAM

        log_P_fine = np.arange(log_P_fine_min, (log_P_fine_max + log_P_fine_step), 
                            log_P_fine_step)

        # Now we can pre-interpolate the sampled opacities (may take up to a minute)
        opac = read_opacities(model, wl, opacity_treatment, T_fine, log_P_fine)


    # Generate our first transmission spectrum
    spectrum = compute_spectrum(planet, star, model, atmosphere, opac, wl,
                                spectrum_type = 'transmission')

    # Write to file
    spectrum_file = write_spectrum(planet_names[i], model_name, spectrum, wl)


    # Add the spectrum we want to plot to an empty spectra plot collection
    if i == 0:
        spectra = plot_collection(spectrum, wl, collection = [])
    else:
        spectra = plot_collection(spectrum, wl, collection = spectra)

In [ ]:
# Produce figure
fig_spec = plot_spectra(spectra, planet, R_to_bin = 100,
                        plot_full_res = False, colour_list = colour_list,
                        spectra_labels = planet_names, plt_label = "Full Range of Masses")

### Make Synthetic Data

In [41]:
data_included = 'NIRISS_G395H_Tiberius'
planet_name = 'TOI-270n'  # Planet name used for plots, output files etc.
model_name = 'H2_H2O_Forward_Model'

data_dir = './data/' + planet_name

datasets = []
instruments = []

if ('NIRISS' in data_included):

    datasets.append(planet_name + '_NIRISS_SOSS_Ord2_ExoTEP.dat')
    datasets.append(planet_name + '_NIRISS_SOSS_Ord1_ExoTEP.dat')
    instruments.append('JWST_NIRISS_SOSS_Ord2')
    instruments.append('JWST_NIRISS_SOSS_Ord1')

if ('G395H_Eureka-feature' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Eureka.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Eureka-CS2.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Eureka' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Eureka.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Eureka.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Tiberius-feature' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Tiberius.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Tiberius-CS2.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Tiberius' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Tiberius.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Tiberius.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')

data = load_data(data_dir, datasets, instruments, wl)

# Specify number of transits observed by each instrument
N_trans = [1, 1, 1, 1]

# Specify spectral resolution for binning each raw dataset
R_to_bin = [None, None, None, None]   # Let's bin down to R = 100 for clarity and retrieval speed

Gauss_scatter = True

In [42]:
for i in range(0,len(planets)):
    model_name = 'H2_H2O_Forward_Model_' + str(i+1)  # Model name used for plots, output files etc.

    # Generate simulated data for the model using errors from data file
    generate_syn_data_from_file(planets[i], wl, spectra[i], data_dir, data, R_to_bin = R_to_bin,
                                N_trans = N_trans, label = model_name, Gauss_scatter = Gauss_scatter)

    if Gauss_scatter == True:
        datasets_new = [planet_names[i] + '_SYNTHETIC_JWST_NIRISS_SOSS_Ord2_' + model_name + '_N_trans_' + str(N_trans[0]) + '.dat',
                        planet_names[i] + '_SYNTHETIC_JWST_NIRISS_SOSS_Ord1_' + model_name + '_N_trans_' + str(N_trans[1]) + '.dat',
                        planet_names[i] + '_SYNTHETIC_JWST_NIRSpec_G395H_NRS1_' + model_name + '_N_trans_' + str(N_trans[2]) + '.dat',
                        planet_names[i] + '_SYNTHETIC_JWST_NIRSpec_G395H_NRS2_' + model_name + '_N_trans_' + str(N_trans[3]) + '.dat']
    else:
        datasets_new = [planet_names[i] + '_SYNTHETIC_JWST_NIRISS_SOSS_Ord2_' + model_name + '_N_trans_' + str(N_trans[0]) + '_no_gauss.dat',
                    planet_names[i] + '_SYNTHETIC_JWST_NIRISS_SOSS_Ord1_' + model_name + '_N_trans_' + str(N_trans[1]) + '_no_gauss.dat',
                    planet_names[i] + '_SYNTHETIC_JWST_NIRSpec_G395H_NRS1_' + model_name + '_N_trans_' + str(N_trans[2]) + '_no_gauss.dat',
                    planet_names[i] + '_SYNTHETIC_JWST_NIRSpec_G395H_NRS2_' + model_name + '_N_trans_' + str(N_trans[3]) + '_no_gauss.dat']

    data_new = load_data(data_dir, datasets_new, instruments, wl)

NameError: name 'spectra' is not defined

### Compute masses, MMWs, and H2O VMRs

In [13]:
masses = np.logspace(np.log10(2),np.log10(20),20)
M_p = masses*M_E
print("Masses:")
print(masses)

Masses:
[ 2.          2.25767578  2.54854997  2.87689978  3.24755348  3.66596142
  4.13827616  4.67144294  5.2733018   5.95270288  6.71963657  7.58538038
  8.5626648   9.66586048 10.91118956 12.31696422 13.90385592 15.69519941
 17.71733581 20.        ]


In [14]:
def exponential(mp, a, b, c, d):
    val = a * np.exp(b * (mp - c)) + d
    val = np.where(val>18, 18, val)
    return val

a = 0.72006201
b = -0.90605788
c = 6.42130887
d = 2.304651282051282

mus = []

print("MMWs:")
for i in range(0,len(masses)):
    mus.append(exponential(masses[i],a,b,c,d))

mus

MMWs:


[array(18.),
 array(18.),
 array(18.),
 array(18.),
 array(15.07579381),
 array(11.04619081),
 array(8.00280281),
 array(5.81972983),
 array(4.34219069),
 array(3.40559331),
 array(2.85416579),
 array(2.55544259),
 array(2.40810713),
 array(2.34272734),
 array(2.31697155),
 array(2.30809838),
 array(2.30546978),
 array(2.30481277),
 array(2.30467713),
 array(2.30465455)]

In [ ]:
# def mean_molecular_weight(H2O_vmr,He_H2_ratio=0.17):
#     """
#     Compute the mean molecular weight of an atmosphere made of H2O, H2, and He,
#     given the volume mixing ratio of H2O.

#     Parameters:
#     -----------
#     H2O_vmr : float
#         Volume mixing ratio of water (between 0 and 1)

#     Returns:
#     --------
#     mu : float
#         Mean molecular weight of the atmosphere in g/mol (amu)
#     """

#     # Remaining VMR for H2 + He
#     remaining_vmr = 1.0 - H2O_vmr

#     # Let x be the VMR of H2
#     # Then He = x * He_H2_ratio
#     # So: x + x * He_H2_ratio = remaining_vmr => x = remaining_vmr / (1 + He_H2_ratio)
#     H2_vmr = remaining_vmr / (1 + He_H2_ratio)
#     He_vmr = H2_vmr * He_H2_ratio

#     # Molecular weights (g/mol)
#     mu_H2O = 18.015
#     mu_H2 = 2.016
#     mu_He = 4.0026

#     # Mean molecular weight: sum(VMR_i * mu_i)
#     mu = (
#             H2O_vmr * mu_H2O +
#             H2_vmr * mu_H2 +
#             He_vmr * mu_He
#     )

#     return mu

# mean_molecular_weight(0.022889740726750180)

2.6642570909319607

In [15]:
def calculate_H2O_vmr(mu,He_H2_ratio=0.17):
    """
    Compute the mean molecular weight of an atmosphere made of H2O, H2, and He,
    given the volume mixing ratio of H2O.

    Parameters:
    -----------

    mu : float
        Mean molecular weight of the atmosphere in g/mol (amu)

    Returns:
    --------
    
    H2O_vmr : float
        Volume mixing ratio of water (between 0 and 1)

    """

    # Molecular weights (g/mol)
    mu_H2O = 18.015
    mu_H2 = 2.016
    mu_He = 4.0026

    # calculate H2O VMR
    # H2O_vmr = (mu - H2_vmr * mu_H2 - He_vmr * mu_He)/mu_H2O
    H2_vmr = (18.015-mu)/18.381108
    H2O_vmr = 1-1.17*H2_vmr

    return H2O_vmr

H2O_VMRs = []
for i in range(0,len(mus)):
    H2O_VMRs.append(calculate_H2O_vmr(mus[i]))

H2O_VMRs

[0.9990452153373997,
 0.9990452153373997,
 0.9990452153373997,
 0.9990452153373997,
 0.8129127342627707,
 0.5564191915334709,
 0.3627005124841022,
 0.22374287219889866,
 0.12969409192758108,
 0.07007750410003388,
 0.03497786799061098,
 0.015963446554507876,
 0.006585203722957567,
 0.0024236293142898546,
 0.0007842134033917425,
 0.0002194156715112916,
 5.209933801997302e-05,
 1.0278880607117458e-05,
 1.6452934263888608e-06,
 2.0798129429788759e-07]

In [19]:
log_X_params = []

print("log_H2O:")
for i in range(0,len(H2O_VMRs)):
    log_X_params.append(np.log10(H2O_VMRs[i]))

print(log_X_params)

log_H2O:
[-0.0004148557908770917, -0.0004148557908770917, -0.0004148557908770917, -0.0004148557908770917, -0.08995607317873024, -0.2545978991089164, -0.440451830775065, -0.6502507911583912, -0.8870798072751445, -1.1544213745181389, -1.4562066656444004, -1.796873337420824, -2.1814307849561487, -2.6155338032635864, -3.105565739223612, -3.6587323566709973, -4.2831677948597715, -4.988054178397378, -5.783756637472411, -6.681975723443587]


In [32]:
data_included = 'NIRISS_G395H_Tiberius'
planet_name = 'TOI-270d'  # Planet name used for plots, output files etc.

data_dir = './data/' + planet_name

datasets = []
instruments = []

if ('NIRISS' in data_included):

    datasets.append(planet_name + '_NIRISS_SOSS_Ord2_ExoTEP.dat')
    datasets.append(planet_name + '_NIRISS_SOSS_Ord1_ExoTEP.dat')
    instruments.append('JWST_NIRISS_SOSS_Ord2')
    instruments.append('JWST_NIRISS_SOSS_Ord1')

if ('G395H_Eureka-feature' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Eureka.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Eureka-CS2.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Eureka' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Eureka.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Eureka.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Tiberius-feature' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Tiberius.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Tiberius-CS2.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')
elif ('G395H_Tiberius' in data_included):

    datasets.append(planet_name + '_NIRSpec_G395H_NRS1_Tiberius.dat',)
    datasets.append(planet_name + '_NIRSpec_G395H_NRS2_Tiberius.dat')
    instruments.append('JWST_NIRSPec_G395H_NRS1')
    instruments.append('JWST_NIRSPec_G395H_NRS2')

data = load_data(data_dir, datasets, instruments, wl)